# Cantonese LoRA parser — dev diagnostics only

This notebook is safe to **Run all**. It never reads the train split, calls no training update, changes no weight, and does not evaluate test. Immediately after Stanza loads the checkpoint, every parameter is frozen and the loader-created optimizer/scheduler references are discarded. It uploads the already-trained result ZIP, verifies the dev-best checkpoint, reconstructs only the fixed dev split, reproduces the frozen predicted POS/lemma cache, and runs two dev inference conditions:

1. frozen predicted POS/lemma (the main protocol condition);
2. gold UPOS/XPOS/FEATS with the same frozen predicted lemma (diagnostic intervention).

Near-similar sentence pairs are manual-review candidates only. No annotation is automatically changed.

In [ ]:
# Install the same software versions as the training run.
%pip install -q stanza==1.14.0 transformers==4.56.2 peft==0.17.1 huggingface-hub==0.34.4 pandas==2.2.3


In [ ]:
from pathlib import Path
import os, json, hashlib, random, shutil, gc, importlib.util, zipfile
import numpy as np
import pandas as pd
import torch
from google.colab import files

SEED=42
WORK=Path('/content/yue_dev_diagnostics_only')
DATA=WORK/'data'; MODELS=WORK/'stanza_resources_1.14.0'; OUT=WORK/'outputs'; HF_HOME=WORK/'hf_home'
for p in (DATA,MODELS,OUT,HF_HOME): p.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(HF_HOME)
os.environ['HF_HUB_DISABLE_XET']='1'

RAW_URL='https://raw.githubusercontent.com/UniversalDependencies/UD_Cantonese-HK/r2.18/yue_hk-ud-test.conllu'
RAW_SHA256='cbd843a195d0db4cdafbf6fcafb7b7b559afea750411006f4728311e70cc4e2a'
DEV_POSITIONS=[4, 12, 50, 59, 67, 69, 75, 95, 114, 120, 148, 153, 163, 168, 174, 182, 222, 224, 225, 232, 242, 262, 268, 281, 283, 285, 286, 288, 337, 338, 348, 410, 412, 416, 427, 430, 432, 434, 453, 463, 470, 494, 498, 504, 530, 536, 546, 567, 574, 579, 595, 599, 610, 627, 628, 640, 653, 672, 685, 686, 688, 693, 701, 716, 726, 728, 729, 731, 734, 765, 766, 768, 769, 776, 792, 794, 798, 803, 816, 817, 824, 835, 837, 854, 871, 887, 905, 914, 931, 932, 936, 941, 960, 961, 963, 975, 979, 987, 997, 999, 1003]
DEV_SHA256='41bc28d903457e70a4747307e56b3eb7820f2e6d3ede10ab549fd4a85aef63b7'
EXPECTED_DEV_CACHE_SHA='e88aa03ff7453469deda68ed30c29dd16769e9839d2010b36abf91faace3bd97'
EXPECTED_CHECKPOINT_SHA='3a56254dca03e20ba77d8fc2310123cce9a129963ff5954c472f0f9c02840f97'
EXPECTED_BEST_DEV=0.7482337829158638
HF_REPO='hfl/chinese-electra-180g-large-discriminator'
HF_REVISION='d017e219578df8e4885484edbc8969dbdea9cbe0'
EVAL_URL='https://universaldependencies.org/conll18/conll18_ud_eval.py'
EVAL_SHA256='1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'
CFG={'batch_size':900}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda','Select a Colab GPU runtime; this notebook does inference only but uses ELECTRA-large.'

print('Upload yue_lora_electra_r8_results.zip')
uploaded=files.upload()
zip_names=[name for name in uploaded if name.endswith('.zip')]
assert len(zip_names)==1,zip_names
result_zip=Path('/content')/zip_names[0]
with zipfile.ZipFile(result_zip) as z:
    member='best_dev_electra_yue_lora_r8.pt'
    assert member in z.namelist()
    z.extract(member,WORK)
best_path=WORK/member
assert sha256(best_path)==EXPECTED_CHECKPOINT_SHA
best_step=1200; best_score=EXPECTED_BEST_DEV
print('checkpoint verified:',sha256(best_path))


In [ ]:
# Reconstruct ONLY dev and verify the project hash.
import urllib.request
raw_path=DATA/'yue_hk-ud-test.r2.18.conllu'
urllib.request.urlretrieve(RAW_URL,raw_path)
assert sha256(raw_path)==RAW_SHA256
raw=raw_path.read_text(encoding='utf-8')
all_blocks=raw.strip().split('\n\n')
assert len(all_blocks)==1004 and len(DEV_POSITIONS)==101
dev_gold=DATA/'dev.conllu'
dev_gold.write_text('\n\n'.join(all_blocks[i-1] for i in DEV_POSITIONS)+'\n\n',encoding='utf-8')
assert sha256(dev_gold)==DEV_SHA256

eval_path=WORK/'conll18_ud_eval.py'
urllib.request.urlretrieve(EVAL_URL,eval_path)
assert sha256(eval_path)==EVAL_SHA256
spec=importlib.util.spec_from_file_location('official_conll18',eval_path)
official=importlib.util.module_from_spec(spec); spec.loader.exec_module(official)
def conll18(gold,system):
    return official.evaluate(official.load_conllu_file(str(gold)),official.load_conllu_file(str(system)))

def split_blocks(text): return text.strip().split('\n\n')
def integer_rows(block):
    return [line.split('\t') for line in block.splitlines()
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit()]


In [ ]:
# Download only inference dependencies at the exact recorded versions.
from huggingface_hub import snapshot_download
snapshot=Path(snapshot_download(HF_REPO,revision=HF_REVISION,cache_dir=HF_HOME/'hub',
    allow_patterns=['config.json','pytorch_model.bin','vocab.txt','tokenizer.json',
                    'tokenizer_config.json','special_tokens_map.json','added_tokens.json']))
assert snapshot.name==HF_REVISION

import stanza
from stanza.resources.common import download_resources_json,load_resources_json
from stanza.pipeline.core import DownloadMethod
assert stanza.__version__=='1.14.0'
download_resources_json(model_dir=str(MODELS))
assert sha256(MODELS/'resources.json')=='4e41c1df152146fa26ed0c006a08feea7a60bb3414bb6d57dbda24ad2e3cb99c'
packages={'tokenize':'gsdsimp','pos':'gsdsimp_electra-large','lemma':'gsdsimp_charlm'}
stanza.download('zh-hans',model_dir=str(MODELS),package=None,processors=packages,verbose=True)

expected={
 'tokenize':'962f2578e2a3dabeb4671053372eb1bd092357921904233556c8d77a46440882',
 'pos':'f7a8cd0ae5c92c07655b7f3e8078d33541f66450dd75541881b6c2740a1b89b4',
 'lemma':'b940e3e3195403228cac8e873e4276ceac4691472310fbd413c344143a59c4ba'}
for proc,pkg in packages.items(): assert sha256(MODELS/'zh-hans'/proc/f'{pkg}.pt')==expected[proc]

os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
tagger=stanza.Pipeline(lang='zh-hans',dir=str(MODELS),processors=packages,
    tokenize_pretokenized=True,use_gpu=True,download_method=DownloadMethod.REUSE_RESOURCES,verbose=False)
for proc in ('pos','lemma'):
    tagger.processors[proc]._trainer.model.eval()
    for p in tagger.processors[proc]._trainer.model.parameters(): p.requires_grad=False

def make_pretagged(src,dst,chunk=32):
    bs=split_blocks(src.read_text(encoding='utf-8')); out=[]
    for start in range(0,len(bs),chunk):
        sub=bs[start:start+chunk]; forms=[[r[1] for r in integer_rows(b)] for b in sub]
        doc=tagger(forms); assert len(doc.sentences)==len(sub)
        for block,sent,gold_forms in zip(sub,doc.sentences,forms):
            assert [w.text for w in sent.words]==gold_forms
            pred=iter(sent.words); lines=[]
            for line in block.splitlines():
                if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                    c=line.split('\t'); w=next(pred)
                    c[2]=w.lemma or '_'; c[3]=w.upos or '_'; c[4]=w.xpos or '_'; c[5]=w.feats or '_'
                    line='\t'.join(c)
                lines.append(line)
            out.append('\n'.join(lines))
    dst.write_text('\n\n'.join(out)+'\n',encoding='utf-8')

dev_predcache=DATA/'dev.predposlemma.conllu'
make_pretagged(dev_gold,dev_predcache)
assert sha256(dev_predcache)==EXPECTED_DEV_CACHE_SHA,(sha256(dev_predcache),EXPECTED_DEV_CACHE_SHA)
del tagger; gc.collect(); torch.cuda.empty_cache()
print('frozen dev cache reproduced:',sha256(dev_predcache))


In [ ]:
# Load the existing checkpoint, then force inference-only state.
from stanza.models.common.pretrain import Pretrain
from stanza.models.depparse.trainer import GraphTrainer
from stanza.models.depparse.data import DataLoader
from stanza.models.depparse.utils import predict_dataset
from stanza.utils.conll import CoNLL
from stanza.models.common.doc import HEAD,DEPREL

resources=load_resources_json(model_dir=str(MODELS))
saved=torch.load(best_path,map_location='cpu',weights_only=True)
dependencies=resources['zh-hans']['pos']['gsdsimp_electra-large'].get('dependencies',[])

def one_local(model_type):
    paths=list((MODELS/'zh-hans'/model_type).glob('*.pt'))
    assert len(paths)==1,(model_type,paths)
    return paths[0].resolve()

load_args={}
if saved['config'].get('charlm'):
    load_args['charlm_forward_file']=str(one_local('forward_charlm'))
    load_args['charlm_backward_file']=str(one_local('backward_charlm'))
pretrain_obj=Pretrain(filename=str(one_local('pretrain'))) if saved['config'].get('pretrain') else None
diag_trainer=GraphTrainer.load(str(best_path),pretrain=pretrain_obj,args=load_args,device=DEVICE)
diag_trainer.model.eval()
for parameter in diag_trainer.model.parameters(): parameter.requires_grad=False
diag_trainer.optimizer=None
diag_trainer.scheduler=None
assert all(not p.requires_grad for p in diag_trainer.model.parameters()), 'Inference model must be fully frozen'

def predict_to_file(tr,loader,path):
    with torch.inference_mode(): preds=predict_dataset(tr,loader)
    loader.doc.set([HEAD,DEPREL],[y for x in preds for y in x])
    path.write_text(f'{loader.doc:C}\n\n',encoding='utf-8')
    return path


## Execute dev diagnostics

The next cell performs only two frozen dev inference passes and exports diagnostic tables. It contains no training update.

In [ ]:
# DEV ONLY: error attribution and gold-POS/morph diagnostic.
import pandas as pd
import unicodedata
from difflib import SequenceMatcher

DIAG=WORK/'dev_diagnostics'; DIAG.mkdir(exist_ok=True)

def make_goldpos_predlemma(pred_cache,gold_path,dst):
    pb=split_blocks(Path(pred_cache).read_text(encoding='utf-8'))
    gb=split_blocks(Path(gold_path).read_text(encoding='utf-8'))
    assert len(pb)==len(gb)
    out=[]
    for pblock,gblock in zip(pb,gb):
        gold_iter=iter(integer_rows(gblock)); lines=[]
        for line in pblock.splitlines():
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                c=line.split('\t'); g=next(gold_iter)
                assert c[0:2]==g[0:2]
                c[3:6]=g[3:6]       # gold UPOS/XPOS/FEATS
                # c[2] remains the frozen predicted lemma
                line='\t'.join(c)
            lines.append(line)
        try: next(gold_iter); raise AssertionError('missing gold word')
        except StopIteration: pass
        out.append('\n'.join(lines))
    dst.write_text('\n\n'.join(out)+'\n',encoding='utf-8')

dev_gold=DATA/'dev.conllu'
dev_predcache=DATA/'dev.predposlemma.conllu'
dev_goldpos=DIAG/'dev.goldpos_predlemma.conllu'
make_goldpos_predlemma(dev_predcache,dev_gold,dev_goldpos)

def run_dev_condition(input_path,output_path):
    doc=CoNLL.conll2doc(input_file=str(input_path))
    loader=DataLoader(doc,CFG['batch_size'],diag_trainer.args,pretrain_obj,
                      vocab=diag_trainer.vocab,evaluation=True,sort_during_eval=True,
                      bert_tokenizer=diag_trainer.model.bert_tokenizer)
    predict_to_file(diag_trainer,loader,output_path)
    return conll18(dev_gold,output_path)

pred_out=DIAG/'dev.predposlemma.best.pred.conllu'
goldpos_out=DIAG/'dev.goldpos_predlemma.best.pred.conllu'
pred_scores=run_dev_condition(dev_predcache,pred_out)
goldpos_scores=run_dev_condition(dev_goldpos,goldpos_out)
assert abs(pred_scores['LAS'].f1-best_score)<1e-12, (pred_scores['LAS'].f1,best_score)

def sentence_meta(block):
    meta={}
    for line in block.splitlines():
        if line.startswith('# ') and ' = ' in line:
            k,v=line[2:].split(' = ',1); meta[k]=v
    return meta

gold_blocks=split_blocks(dev_gold.read_text(encoding='utf-8'))
input_blocks=split_blocks(dev_predcache.read_text(encoding='utf-8'))
pred_blocks=split_blocks(pred_out.read_text(encoding='utf-8'))
oracle_blocks=split_blocks(goldpos_out.read_text(encoding='utf-8'))
records=[]; sent_records=[]
for si,(gb,ib,pb,ob) in enumerate(zip(gold_blocks,input_blocks,pred_blocks,oracle_blocks),1):
    gr,ir,pr,orr=map(integer_rows,(gb,ib,pb,ob)); meta=sentence_meta(gb)
    assert len(gr)==len(ir)==len(pr)==len(orr)
    correct=0
    for g,inp,p,o in zip(gr,ir,pr,orr):
        assert g[0:2]==inp[0:2]==p[0:2]==o[0:2]
        gh,ph,oh=int(g[6]),int(p[6]),int(o[6]); gd,pd,od=g[7],p[7],o[7]
        base=lambda x:x.split(':',1)[0]
        dep_len=0 if gh==0 else abs(int(g[0])-gh)
        if gh==0: bucket='ROOT'
        elif dep_len==1: bucket='1'
        elif dep_len==2: bucket='2'
        elif dep_len<=5: bucket='3-5'
        elif dep_len<=10: bucket='6-10'
        else: bucket='11+'
        las=(gh==ph and base(gd)==base(pd)); correct+=las
        records.append(dict(sentence_index=si,sent_id=meta.get('sent_id'),text=meta.get('text'),
            token_id=int(g[0]),form=g[1],gold_upos=g[3],predicted_upos=inp[3],
            upos_correct=g[3]==inp[3],full_morph_correct=g[3:6]==inp[3:6],
            gold_head=gh,pred_head=ph,goldpos_pred_head=oh,gold_deprel=gd,
            pred_deprel=pd,goldpos_pred_deprel=od,head_distance=dep_len,distance_bucket=bucket,
            uas=gh==ph,conll18_las=las,strict_las=gh==ph and gd==pd,
            goldpos_uas=gh==oh,goldpos_conll18_las=gh==oh and base(gd)==base(od),
            goldpos_strict_las=gh==oh and gd==od))
    sent_records.append(dict(sentence_index=si,sent_id=meta.get('sent_id'),text=meta.get('text'),
                             tokens=len(gr),conll18_las=correct/len(gr)))

# The loop above uses `pd` as a short-lived predicted-DEPREL variable.
# Restore the pandas module alias before constructing tables.
import pandas as pd
df=pd.DataFrame(records); sdf=pd.DataFrame(sent_records)
df.to_csv(DIAG/'token_diagnostics.csv',index=False)
sdf.sort_values(['conll18_las','tokens']).to_csv(DIAG/'worst_sentences.csv',index=False)

def grouped_stats(column):
    return (df.groupby(column,dropna=False)
      .agg(tokens=('token_id','size'),upos_accuracy=('upos_correct','mean'),
           UAS=('uas','mean'),CoNLL18_LAS=('conll18_las','mean'),strict_LAS=('strict_las','mean'),
           goldPOS_CoNLL18_LAS=('goldpos_conll18_las','mean')).reset_index())

distance_stats=grouped_stats('distance_bucket')
distance_stats['distance_bucket']=pd.Categorical(distance_stats['distance_bucket'],['ROOT','1','2','3-5','6-10','11+'],ordered=True)
distance_stats.sort_values('distance_bucket').to_csv(DIAG/'distance_stats.csv',index=False)
relation_stats=grouped_stats('gold_deprel').sort_values(['CoNLL18_LAS','tokens'],ascending=[True,False])
relation_stats.to_csv(DIAG/'relation_stats.csv',index=False)
upos_stats=grouped_stats('upos_correct'); upos_stats.to_csv(DIAG/'pos_error_association.csv',index=False)

conf=(df[df['uas'] & ~df['strict_las']]
      .groupby(['gold_deprel','pred_deprel']).size().reset_index(name='count')
      .sort_values('count',ascending=False))
conf.to_csv(DIAG/'relation_confusions_when_head_correct.csv',index=False)

# Exact duplicates and annotation disagreements inside dev.
groups={}
for i,b in enumerate(gold_blocks):
    r=integer_rows(b); forms=tuple(unicodedata.normalize('NFC',x[1]) for x in r)
    joined=unicodedata.normalize('NFC',''.join(x[1] for x in r))
    for key in (('tokens',forms),('joined',joined)): groups.setdefault(key,set()).add(i)
dup=[]
for key,idxs in groups.items():
    if len(idxs)<2: continue
    analyses={tuple((x[6],x[7]) for x in integer_rows(gold_blocks[i])) for i in idxs}
    dup.append({'key_type':key[0],'indices_1_based':[i+1 for i in sorted(idxs)],
                'sent_ids':[sentence_meta(gold_blocks[i]).get('sent_id') for i in sorted(idxs)],
                'annotation_disagreement':len(analyses)>1})
(DIAG/'exact_duplicate_annotation_audit.json').write_text(json.dumps(dup,ensure_ascii=False,indent=2),encoding='utf-8')

# Near-similar forms are candidates for manual review only.
near=[]
texts=[''.join(x[1] for x in integer_rows(b)) for b in gold_blocks]
for i in range(len(texts)):
    for j in range(i+1,len(texts)):
        ratio=SequenceMatcher(None,texts[i],texts[j],autojunk=False).ratio()
        if ratio>=0.85 and texts[i]!=texts[j]:
            near.append({'similarity':ratio,'index_a':i+1,'index_b':j+1,
                         'sent_id_a':sentence_meta(gold_blocks[i]).get('sent_id'),
                         'sent_id_b':sentence_meta(gold_blocks[j]).get('sent_id'),
                         'text_a':sentence_meta(gold_blocks[i]).get('text'),
                         'text_b':sentence_meta(gold_blocks[j]).get('text')})
pd.DataFrame(near).sort_values('similarity',ascending=False).to_csv(DIAG/'near_similar_manual_review_candidates.csv',index=False) if near else (DIAG/'near_similar_manual_review_candidates.csv').write_text('similarity,index_a,index_b,sent_id_a,sent_id_b,text_a,text_b\n')

def pct(x): return 100*float(x)
pos_wrong=df[~df.upos_correct]; pos_right=df[df.upos_correct]
summary={
 'split':'dev','checkpoint_selected_without_test':True,'best_step':best_step,
 'predicted_condition':{'UAS_percent':pct(pred_scores['UAS'].f1),'CoNLL18_LAS_percent':pct(pred_scores['LAS'].f1),
                        'strict_LAS_percent':pct(df.strict_las.mean())},
 'gold_pos_morph_predicted_lemma_condition':{'UAS_percent':pct(goldpos_scores['UAS'].f1),
                        'CoNLL18_LAS_percent':pct(goldpos_scores['LAS'].f1),
                        'strict_LAS_percent':pct(df.goldpos_strict_las.mean())},
 'gold_pos_delta_points':pct(goldpos_scores['LAS'].f1-pred_scores['LAS'].f1),
 'predicted_UPOS_accuracy_percent':pct(df.upos_correct.mean()),
 'dependency_LAS_given_UPOS_correct_percent':pct(pos_right.conll18_las.mean()),
 'dependency_LAS_given_UPOS_wrong_percent':pct(pos_wrong.conll18_las.mean()),
 'tokens_UPOS_correct':len(pos_right),'tokens_UPOS_wrong':len(pos_wrong),
 'exact_duplicate_groups':len(dup),
 'exact_duplicate_groups_with_annotation_disagreement':sum(x['annotation_disagreement'] for x in dup),
 'near_similar_pairs_for_manual_review':len(near),
 'interpretation_warning':'Gold POS/morph is a diagnostic distribution shift because training used predicted tags. Near-similar pairs are not automatically annotation errors.'}
(DIAG/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(upos_stats, distance_stats, relation_stats.head(20), conf.head(20), sdf.sort_values('conll18_las').head(15))
diag_zip=shutil.make_archive('/content/yue_dev_diagnostics','zip',root_dir=DIAG)
from google.colab import files
files.download(diag_zip)
